# NB06C — Post-Pretraining Diagnostics

Reads NB06 training logs, checks checkpoint integrity, and computes pass/warn/fail gates. Includes a backbone-was-actually-trained sanity gate that compares the pretrained ConvNeXt-Tiny weights to the ImageNet baseline; weights identical to ImageNet would indicate Phase 2 did not run.

In [ ]:
import os, json, hashlib, shutil, tempfile
from pathlib import Path
from datetime import datetime

WORKSPACE = Path(os.environ.get('WORKSPACE', './workspace'))
LOGS_DIR        = WORKSPACE / 'logs'
MODELS_DIR      = WORKSPACE / 'weights'
FEATURES_05_DIR = WORKSPACE / 'features' / 'scale0p5'
FEATURES_20_DIR = WORKSPACE / 'features' / 'scale2p0'
DIAG_DIR        = WORKSPACE / 'diagnostics'
DIAG_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_LOG_CSV = LOGS_DIR / 'nb06_train_log.csv'

import pandas as pd
import numpy as np

try:
    import torch
    HAS_TORCH = True
except Exception:
    HAS_TORCH = False

try:
    from safetensors.torch import load_file as load_safetensors
    HAS_ST = True
except Exception:
    HAS_ST = False

try:
    import torchvision.models as tvm
    HAS_TV = True
except Exception:
    HAS_TV = False

def safe_read_csv(path: Path) -> pd.DataFrame:
    if not path.exists(): return pd.DataFrame()
    with tempfile.NamedTemporaryFile(delete=False, suffix='.csv') as tmp:
        tmp_path = Path(tmp.name)
    try:
        shutil.copy2(path, tmp_path)
        df = pd.read_csv(tmp_path)
    except Exception:
        df = pd.DataFrame()
    finally:
        try: tmp_path.unlink(missing_ok=True)
        except Exception: pass
    return df

def list_ckpts(models_dir: Path):
    exts = ('.pt', '.pth', '.safetensors')
    return sorted([p for p in models_dir.glob('*') if p.suffix.lower() in exts],
                  key=lambda x: x.stat().st_mtime)

def sha256_12(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()[:12]

def try_load(path: Path):
    if path.suffix == '.safetensors':
        if not HAS_ST:
            return False, {'error': 'safetensors not available'}
        try:
            obj = load_safetensors(str(path))
            return True, {'type': 'safetensors_dict', 'top_keys': list(obj.keys())[:8]}
        except Exception as e:
            return False, {'error': str(e)[:180]}
    if not HAS_TORCH:
        return False, {'error': 'torch not available'}
    try:
        obj = torch.load(path, map_location='cpu', weights_only=False)
        meta = {'type': type(obj).__name__}
        if isinstance(obj, dict): meta['top_keys'] = list(obj.keys())[:8]
        return True, meta
    except Exception as e:
        return False, {'error': str(e)[:180]}

def count_2scale_slides():
    s05 = {p.stem for p in FEATURES_05_DIR.glob('*.npy')}
    s20 = {p.stem for p in FEATURES_20_DIR.glob('*.npy')}
    return len(s05 & s20), len(s05), len(s20)

def imagenet_param_norms() -> dict:
    if not (HAS_TORCH and HAS_TV):
        return {}
    w = tvm.ConvNeXt_Tiny_Weights.DEFAULT
    m = tvm.convnext_tiny(weights=w)
    feats_sd = m.features.state_dict()
    return {k: float(v.float().norm().item()) for k, v in feats_sd.items()}

df_log = safe_read_csv(TRAIN_LOG_CSV)
diag = {
    'time': datetime.now().isoformat(timespec='seconds'),
    'workspace': str(WORKSPACE),
    'log_csv_exists': TRAIN_LOG_CSV.exists(),
    'log_rows': int(len(df_log)),
}

for c in ['phase', 'epoch', 'step', 'loss', 'loss_byol', 'loss_mfr', 'tps', 'vram_gb', 'ts']:
    if c not in df_log.columns:
        df_log[c] = datetime.now().isoformat(timespec='seconds') if c == 'ts' else np.nan

for c in ['phase', 'epoch', 'step', 'loss', 'loss_byol', 'loss_mfr', 'tps', 'vram_gb']:
    df_log[c] = pd.to_numeric(df_log[c], errors='coerce')

diag['steps_logged'] = int(df_log['step'].max()) if len(df_log) else 0

if len(df_log) > 3 and df_log['loss'].notna().any():
    n = len(df_log)
    head = df_log['loss'].dropna().iloc[:max(3, n//10)]
    tail = df_log['loss'].dropna().iloc[-max(3, n//10):]
    start_med = float(np.median(head)) if len(head) else np.nan
    end_med   = float(np.median(tail)) if len(tail) else np.nan
    rel_impr  = float((start_med - end_med) / start_med) if (start_med and start_med == start_med) else np.nan
else:
    start_med = end_med = rel_impr = np.nan

diag['loss_start_median'] = start_med
diag['loss_end_median']   = end_med
diag['loss_rel_improvement'] = rel_impr

n_both, n05, n20 = count_2scale_slides()
diag['features_2scale_intersection'] = int(n_both)

ckpts = list_ckpts(MODELS_DIR)
diag['checkpoint_count'] = int(len(ckpts))
ckpt_info = []
for p in ckpts[-6:]:
    ok, meta = try_load(p)
    ckpt_info.append({
        'file': str(p), 'size_mb': round(p.stat().st_size/(1024**2), 2),
        'sha256_12': sha256_12(p), 'load_ok': bool(ok), 'meta': meta,
    })
diag['checkpoints_recent'] = ckpt_info
diag['suggest_checkpoint'] = (str(ckpts[-1]) if len(ckpts) else None)

backbone_check = {'available': False, 'note': 'no backbone checkpoint found'}
backbone_ckpts = [p for p in ckpts if 'backbone' in p.name.lower()]
if backbone_ckpts and HAS_ST and HAS_TV and HAS_TORCH:
    try:
        latest_bb = backbone_ckpts[-1]
        sd = load_safetensors(str(latest_bb))
        imnet_norms = imagenet_param_norms()
        if imnet_norms:
            n_total = 0; n_changed = 0
            max_rel = 0.0; total_rel = 0.0
            for k, v in sd.items():
                k_clean = k.replace('features.', '') if k.startswith('features.') else k
                ref = imnet_norms.get(k) or imnet_norms.get(k_clean) or imnet_norms.get('features.' + k)
                if ref is None: continue
                trained_norm = float(v.float().norm().item())
                rel = abs(trained_norm - ref) / max(ref, 1e-6)
                n_total += 1
                if rel > 1e-4:
                    n_changed += 1
                max_rel = max(max_rel, rel)
                total_rel += rel
            backbone_check = {
                'available': True,
                'checkpoint': str(latest_bb),
                'n_params_compared': n_total,
                'n_params_changed': n_changed,
                'mean_rel_change': total_rel / max(n_total, 1),
                'max_rel_change': max_rel,
                'verdict': 'TRAINED' if n_changed > 0.5 * n_total else 'LIKELY_FROZEN',
            }
    except Exception as e:
        backbone_check = {'available': False, 'note': f'check failed: {e}'}
diag['backbone_check'] = backbone_check

gates = []
if n_both >= 18000:
    gates.append(('G1_2scale_coverage', 'PASS', f'{n_both} slides with both scales'))
elif n_both >= 15000:
    gates.append(('G1_2scale_coverage', 'WARN', f'{n_both} < expected'))
else:
    gates.append(('G1_2scale_coverage', 'FAIL', f'{n_both} very low'))

if rel_impr == rel_impr:
    if rel_impr >= 0.60:   gates.append(('G2_loss_improvement', 'PASS', f'relative drop {rel_impr:.2f}'))
    elif rel_impr >= 0.30: gates.append(('G2_loss_improvement', 'WARN', f'modest drop {rel_impr:.2f}'))
    else:                  gates.append(('G2_loss_improvement', 'FAIL', f'weak drop {rel_impr:.2f}'))
else:
    gates.append(('G2_loss_improvement', 'WARN', 'loss trend unavailable'))

if not ckpts:
    gates.append(('G3_checkpoints', 'FAIL', 'no model files'))
elif any(not c['load_ok'] for c in ckpt_info):
    gates.append(('G3_checkpoints', 'WARN', f"{sum(1 for c in ckpt_info if not c['load_ok'])} failed to load"))
else:
    gates.append(('G3_checkpoints', 'PASS', f'{len(ckpts)} file(s), latest loads OK'))

if backbone_check.get('available'):
    if backbone_check['verdict'] == 'TRAINED':
        gates.append(('G4_backbone_trained', 'PASS',
                      f"{backbone_check['n_params_changed']}/{backbone_check['n_params_compared']} params changed; mean rel \u0394={backbone_check['mean_rel_change']:.4f}"))
    else:
        gates.append(('G4_backbone_trained', 'FAIL',
                      'backbone weights nearly identical to ImageNet baseline'))
else:
    gates.append(('G4_backbone_trained', 'WARN', backbone_check.get('note', 'unavailable')))

diag['gates'] = [{'name': n, 'status': s, 'detail': d} for (n, s, d) in gates]

DIAG_JSON = DIAG_DIR / 'nb06c_posttrain_diagnostics.json'
with open(DIAG_JSON, 'w', encoding='utf-8') as f:
    json.dump(diag, f, indent=2, ensure_ascii=False)

print('\nGATES:')
for (n, s, d) in gates:
    tag = {'PASS': '[ OK ]', 'WARN': '[WARN]', 'FAIL': '[FAIL]'}[s]
    print(f' {tag} {n}: {d}')
print(f'[OK] diagnostics saved: {DIAG_JSON}')